In [ ]:
# %pip install yt_dlp
# %pip install -q dvt

In [ ]:
import os
import av
import cv2
import numpy as np
from yt_dlp import YoutubeDL

TEMP_DIR = './temp_video'

def download_video(url):
    """1. Downloads the video and returns the actual file path."""
    if not os.path.exists(TEMP_DIR):
        os.makedirs(TEMP_DIR)
        
    ydl_opts = {
        'outtmpl': os.path.join(TEMP_DIR, 'input.%(ext)s'),
        'format': 'bestvideo[height<=720][ext=mp4]/best[ext=mp4]',
        'quiet': True,
        'no_warnings': True
    }

    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return ydl.prepare_filename(info)


def cleanup(file_path):
    """4. Deletes the file if it exists."""
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"Cleanup: Deleted {file_path}")




In [ ]:
target_url = 'https://www.youtube.com/watch?v=Unfpbdbo3hY'
vid_filename = download_video(target_url)
vid_filename


In [ ]:
import dvt
import polars as pl
from PIL import Image

meta = dvt.video_info(vid_filename)
meta

In [ ]:
from dvt_mod.annotate_mod import annotate_shots

output = annotate_shots(vid_filename)

In [ ]:
dt = pl.from_dict(output['scenes'])
dt = dt.with_columns((pl.col("start") / meta['fps']).alias("start_time"))
dt = dt.with_columns((pl.col("end") / meta['fps']).alias("end_time"))
dt

In [ ]:
from dvt_mod.annotate_mod import yield_video

first_frames = []
for image, frame, timestamp in yield_video(vid_filename):
    if frame in dt['start'].to_list():
        first_frames += [frame]
        pimg = Image.fromarray(image, 'RGB')
        display(pimg)

In [ ]:
cleanup(vid_filename)